# LeetCode 3929 — Minimum Partition Score II bằng Python

- Link: https://leetcode.com/problems/minimum-partition-score-ii/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động chia đoạn, hàm lồi, tối ưu chia để trị (divide & conquer optimization), **Aliens trick** (Lagrange / WQS binary search), Convex Hull Trick

> **Bài Premium: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**
> Bài Premium không có skeleton công khai. Tên phương thức `minPartitionScore(self, nums, k)` được suy luận — **cần đối chiếu** lại chữ ký trên LeetCode khi nộp.

Skeleton (suy luận):

```python
class Solution:
    def minPartitionScore(self, nums: List[int], k: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Chia mảng `nums` (toàn số **dương**) thành **đúng `k`** mảng con liên tiếp. Giá trị của một mảng con có tổng `s` là `s·(s+1)/2`. Điểm của một cách chia là tổng giá trị các mảng con. Tìm điểm **nhỏ nhất**.

Quy hoạch động thông thường `dp[t][i]` (chia `i` phần tử đầu thành `t` đoạn) tốn `O(k·n²)` — với `n = 5·10^4` là quá chậm. Điểm mấu chốt: chi phí `s²` là hàm **lồi** nên đáp án `F(k)` (điểm tối thiểu khi dùng `k` đoạn) là hàm **lồi theo `k`**. Hàm lồi cho phép **Aliens trick**: thay ràng buộc "đúng `k` đoạn" bằng một **hình phạt `λ` cho mỗi đoạn**, rồi tìm nhị phân `λ`. Bài toán còn lại (tối thiểu hóa với phạt) giải bằng DP một chiều tăng tốc bằng **Convex Hull Trick** (CHT) trong `O(n)`.

Sau notebook này bạn sẽ:

1. Rút gọn điểm số: `Σ s(s+1)/2 = (Σ s² + tổng cả mảng) / 2`, nên chỉ cần tối thiểu `Σ s²`.
2. Viết DP chia đoạn `O(k·n²)`, rồi tăng tốc bằng **tối ưu chia để trị** `O(k·n·log n)`.
3. Hiểu **Aliens trick**: vì sao `F` lồi giúp "đổi ràng buộc số đoạn lấy một hình phạt `λ`", cách tìm `λ` bằng nhị phân, cách khôi phục `F(k) = h(λ) − λ·k`, và cách **xử lý hòa** (nhiều số đoạn cùng tối ưu).
4. Cài **Convex Hull Trick** với số nguyên lớn (không dùng số thực) trong `O(n)` cho mỗi `λ`.
5. Viết 5 lời giải (vét cạn, DP `O(kn²)`, chia để trị, Aliens + DP bình phương, Aliens + CHT) và đối chiếu bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums` (mọi phần tử dương) và `k`. Chia `nums` thành **đúng `k`** mảng con liên tiếp, không rỗng, phủ hết mảng. Giá trị mảng con có tổng `sumArr` là `sumArr·(sumArr + 1)/2`. Điểm của cách chia = tổng giá trị. Trả về điểm nhỏ nhất.

**Ví dụ 1:** `nums = [5,1,2,1]`, `k = 2` → `25` (`[5]` có giá trị 15, `[1,2,1]` tổng 4 giá trị 10).

**Ví dụ 2:** `nums = [1,2,3,4]`, `k = 1` → `55` (`10·11/2`).

**Ví dụ 3:** `nums = [1,1,1]`, `k = 3` → `3`.

**Ràng buộc:** `1 ≤ len(nums) ≤ 5·10^4`, `1 ≤ nums[i] ≤ 10^3`, `1 ≤ k ≤ len(nums)`.

In [1]:
import random, time
from itertools import combinations

EXAMPLES = [
    (([5, 1, 2, 1], 2), 25),
    (([1, 2, 3, 4], 1), 55),
    (([1, 1, 1], 3), 3),
]
for args, exp in EXAMPLES:
    print(args, "-> kỳ vọng", exp)

([5, 1, 2, 1], 2) -> kỳ vọng 25
([1, 2, 3, 4], 1) -> kỳ vọng 55
([1, 1, 1], 3) -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Tổng tiền tố (prefix sums)

`S[0] = 0`, `S[i] = nums[0] + ... + nums[i−1]`. Tổng đoạn `nums[j:i]` (nửa mở) là `S[i] − S[j]`. Vì mọi `nums[i] > 0`, `S` **tăng ngặt**.

In [2]:
nums = [5, 1, 2, 1]
S = [0]
for x in nums:
    S.append(S[-1] + x)
print(S)                       # [0, 5, 6, 8, 9]
print("tổng nums[1:4] =", S[4] - S[1])   # 1+2+1 = 4

[0, 5, 6, 8, 9]
tổng nums[1:4] = 4


### 3.2. Điểm = `(Σ s² + tổng) / 2`

`s(s+1)/2 = (s² + s)/2`. Tổng các `s` của mọi đoạn bằng **tổng cả mảng** `T` (không đổi dù chia thế nào). Vậy:

```
điểm = ( Σ s² + T ) / 2
```

Tối thiểu điểm ⇔ tối thiểu `Σ s²` với **đúng `k` đoạn**. Ví dụ 1: `[5] | [1,2,1]` → `Σ s² = 25 + 16 = 41`, `T = 9`, điểm `(41 + 9)/2 = 25` ✓.

In [3]:
def score_direct(parts):
    return sum(sum(p) * (sum(p) + 1) // 2 for p in parts)

def score_via_squares(parts):
    sq = sum(sum(p) ** 2 for p in parts)
    total = sum(sum(p) for p in parts)
    return (sq + total) // 2

parts = [[5], [1, 2, 1]]
print(score_direct(parts), score_via_squares(parts))
assert score_direct(parts) == score_via_squares(parts) == 25

25 25


### 3.3. DP chia đoạn

`dp[t][i]` = `Σ s²` nhỏ nhất khi chia `i` phần tử đầu thành đúng `t` đoạn:

```
dp[t][i] = min_{t−1 ≤ j < i}  dp[t−1][j] + (S[i] − S[j])²
dp[0][0] = 0
```

Đáp án `(dp[k][n] + S[n]) / 2`. Đây là `O(k·n²)`.

### 3.4. Hàm lồi và độ dốc

Hàm `F(m)` gọi là **lồi** nếu các hiệu `F(m) − F(m+1)` (độ giảm khi thêm một đoạn) **không tăng** theo `m`: thêm đoạn đầu tiên giúp nhiều nhất, thêm đoạn sau giúp ít dần. Với `F(m) = min Σ s²` điều này đúng vì hàm chi phí đoạn `s²` thỏa **bất đẳng thức tứ giác** (Monge), một hệ quả quen thuộc của việc `x²` là hàm lồi. Ta sẽ kiểm chứng bằng số ở mục 6.

In [4]:
def dp_all_k(nums):
    # F[m] = min tổng bình phương khi chia thành đúng m đoạn (m = 1..n), O(n^3), chỉ để quan sát
    n = len(nums)
    S = [0]
    for x in nums:
        S.append(S[-1] + x)
    INF = float("inf")
    dp = [0] + [INF] * n
    F = [None]
    for t in range(1, n + 1):
        nd = [INF] * (n + 1)
        for i in range(t, n + 1):
            nd[i] = min(dp[j] + (S[i] - S[j]) ** 2 for j in range(t - 1, i))
        dp = nd
        F.append(dp[n])
    return F

F = dp_all_k([3, 1, 4, 1, 5, 9, 2, 6])
print("F(m):", F[1:])
print("độ giảm F(m) - F(m+1):", [F[m] - F[m + 1] for m in range(1, len(F) - 1)])

F(m): [961, 485, 341, 245, 211, 187, 179, 173]
độ giảm F(m) - F(m+1): [476, 144, 96, 34, 24, 8, 6]


### 3.5. Đường thẳng và bao dưới (Convex Hull Trick)

Biểu thức `dp[j] + (S[i] − S[j])² = S[i]² + (dp[j] + S[j]²) − 2·S[j]·S[i]`. Với `i` thay đổi, mỗi `j` là một **đường thẳng** theo biến `x = S[i]`:

```
đường thẳng j :  y = (−2·S[j]) · x + (dp[j] + S[j]²)
```

Ta cần giá trị nhỏ nhất trong tất cả đường thẳng đã thêm tại điểm `x = S[i]`. Các đường thẳng thêm vào theo **độ dốc giảm dần** (`S[j]` tăng) và các truy vấn `x = S[i]` **tăng dần** ⇒ dùng một hàng đợi hai đầu: loại các đường thẳng không bao giờ là nhỏ nhất (không nằm trên **bao dưới**) khi thêm; con trỏ truy vấn chỉ tiến về phía sau.

### 3.6. Số nguyên lớn thay cho số thực

`S[n]² ≈ (5·10^7)² = 2.5·10^15`, vượt khả năng biểu diễn **chính xác** của `float` (`2^53 ≈ 9·10^15` còn đúng cho số nguyên nhưng phép chia/giao điểm sai số lớn). Ta so sánh các giao điểm bằng **nhân chéo** trên số nguyên Python (độ chính xác tùy ý).

In [5]:
print(float(2**53 + 1) == float(2**53))        # True: float mất chính xác ở 2^53
# so sánh phân số a/b <= c/d chính xác bằng nhân chéo (b, d > 0)
a, b, c, d = 10**15 + 1, 3, 10**15, 3
print(a * d <= c * b)                           # False, đúng: (10^15+1)/3 > 10^15/3

True
False


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `nums = [5,1,2,1]`, `k = 2`. Có `3` cách chia thành 2 đoạn:

| Cách chia | `s` | `Σ s²` | điểm `(Σ s² + 9)/2` |
|---|---|---|---|
| `[5] \| [1,2,1]` | 5, 4 | 41 | 25 |
| `[5,1] \| [2,1]` | 6, 3 | 45 | 27 |
| `[5,1,2] \| [1]` | 8, 1 | 65 | 37 |

Nhỏ nhất `25`. Quan sát: `Σ s²` nhỏ nhất khi các `s` **đều nhau nhất** (do `x²` lồi) — chia thành các đoạn có tổng gần bằng nhau.

Nhưng đề yêu cầu **đúng `k`** đoạn; vì `Σ s²` luôn **giảm** khi tách đoạn (mọi phần tử dương), ta không thể bỏ `k` đi: không có ràng buộc thì đáp án là `n` đoạn đơn lẻ. Ràng buộc "đúng `k`" là phần khó.

In [6]:
nums, k = [5, 1, 2, 1], 2
n = len(nums)
best = None
for cuts in combinations(range(1, n), k - 1):
    bounds = (0,) + cuts + (n,)
    parts = [nums[bounds[i]:bounds[i + 1]] for i in range(k)]
    sq = sum(sum(p) ** 2 for p in parts)
    print(parts, "Σ s² =", sq, "điểm =", (sq + sum(nums)) // 2)

[[5], [1, 2, 1]] Σ s² = 41 điểm = 25
[[5, 1], [2, 1]] Σ s² = 45 điểm = 27
[[5, 1, 2], [1]] Σ s² = 65 điểm = 37


## 5. Cách 1 — Brute Force: thử mọi cách chọn điểm cắt

**Ý tưởng.** Chọn `k − 1` điểm cắt trong `n − 1` khe giữa các phần tử (`C(n−1, k−1)` cách), tính điểm bằng định nghĩa, lấy `min`.

In [7]:
def brute(nums, k):
    n = len(nums)
    best = None
    for cuts in combinations(range(1, n), k - 1):
        bounds = (0,) + cuts + (n,)
        score = 0
        for i in range(k):
            s = sum(nums[bounds[i]:bounds[i + 1]])
            score += s * (s + 1) // 2
        if best is None or score < best:
            best = score
    return best

for args, exp in EXAMPLES:
    got = brute(*args)
    print(args, "->", got)
    assert got == exp

([5, 1, 2, 1], 2) -> 25
([1, 2, 3, 4], 1) -> 55
([1, 1, 1], 3) -> 3


**Độ phức tạp.** `O(C(n−1, k−1) · n)`: dùng được tới `n ≈ 18`. **Nút thắt:** số cách chia bùng nổ; cần quy hoạch động.

## 6. Bước đột phá

**Bước 1 — DP chia đoạn** (mục 3.3): `O(k·n²)`.

**Bước 2 — Tính lồi theo `k`.** Với mỗi `m`, `F(m) = min Σ s²` khi chia đúng `m` đoạn. Cell kiểm chứng ở mục 3.4 cho thấy độ giảm `F(m) − F(m+1)` không tăng. Đây là tính chất cốt lõi: `F` lồi **và giảm**.

**Bước 3 — Aliens trick (phạt Lagrange).** Với mỗi hệ số phạt `λ ≥ 0`, xét bài toán **không ràng buộc số đoạn**:

```
h(λ) = min_m [ F(m) + λ·m ]
```

Chọn `m` đoạn phải trả thêm `λ` cho mỗi đoạn. `λ` lớn → ít đoạn; `λ = 0` → nhiều đoạn nhất (`n`). Vì `F` lồi, với mỗi `m` tồn tại một khoảng `λ` mà `m` là số đoạn tối ưu — đó là các `λ` nằm giữa hai độ giảm liên tiếp `F(m−1) − F(m) ≥ λ ≥ F(m) − F(m+1)`.

**Nếu biết `λ` làm cho `k` là số đoạn tối ưu thì `F(k) = h(λ) − λ·k`.** Ta tìm `λ` bằng nhị phân trên số nguyên vì độ giảm đều là số nguyên.

**Xử lý hòa.** Với một `λ`, nhiều `m` có thể cùng tối ưu (đúng khi `λ` bằng một độ giảm). Khi giải DP có phạt, ta luôn chọn **số đoạn nhỏ nhất** trong các phương án cùng giá: gọi là `cnt(λ)`. Hàm `cnt(λ)` **không tăng** theo `λ`. Ta tìm `λ` **nhỏ nhất** sao cho `cnt(λ) ≤ k`; khi đó `k` thuộc khoảng các số đoạn tối ưu của `λ` (vì ở `λ − 1`, số đoạn nhỏ nhất đã `> k`, mà các độ giảm là số nguyên) và `F(k) = h(λ) − λ·k`.

**Bước 4 — DP có phạt trong `O(n)`.** `dp[i] = min_j dp[j] + (S[i] − S[j])² + λ` là DP một chiều có dạng đường thẳng nên dùng CHT (mục 3.5) với truy vấn đơn điệu → `O(n)` cho mỗi `λ`. Hòa được xử lý bằng cách **ghép đếm vào giá trị**: dùng khóa `dp·W + cnt` với `W > n` để so sánh theo `(giá trị, số đoạn)` bằng một số nguyên duy nhất.

**Tổng:** `O(n · log(S²)) ≈ 5·10^4 · 52 ≈ 2.6·10^6` bước CHT.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP chia đoạn `O(k·n²)`

**Giải thích từng đoạn.** Hai lớp `prev` (`dp[t−1]`) và `cur` (`dp[t]`). Với mỗi `i` trong `[t, n]`, `cur[i] = min_{j ∈ [t−1, i−1]} prev[j] + (S[i] − S[j])²`. Sau `k` lớp, đáp án `(dp[k][n] + S[n])/2`.

In [8]:
def prefix_sums(nums):
    S = [0]
    for x in nums:
        S.append(S[-1] + x)
    return S

def dp_kn2(nums, k):
    n = len(nums)
    S = prefix_sums(nums)
    INF = float("inf")
    prev = [0] + [INF] * n
    for t in range(1, k + 1):
        cur = [INF] * (n + 1)
        for i in range(t, n + 1):
            cur[i] = min(prev[j] + (S[i] - S[j]) ** 2 for j in range(t - 1, i))
        prev = cur
    return (prev[n] + S[n]) // 2

for args, exp in EXAMPLES:
    got = dp_kn2(*args)
    print(args, "->", got)
    assert got == exp

([5, 1, 2, 1], 2) -> 25
([1, 2, 3, 4], 1) -> 55
([1, 1, 1], 3) -> 3


**Độ phức tạp:** `O(k·n²)` thời gian, `O(n)` không gian (hai lớp). Với `n = 5·10^4`, `k = 25000`: không khả thi.

### 7.2. Cách 3 — Tối ưu chia để trị (divide & conquer optimization)

**Ý tưởng.** Với cost thỏa bất đẳng thức tứ giác, **điểm chia tốt nhất** `opt[i]` của `dp[t][i]` **không giảm** khi `i` tăng. Nên để tính cả lớp `cur[lo..hi]`: lấy `mid`, tìm `opt[mid]` bằng cách thử mọi `j` trong `[optlo, opthi]`; rồi đệ quy: nửa trái dùng `[optlo, opt[mid]]`, nửa phải dùng `[opt[mid], opthi]`. Mỗi lớp `O(n log n)`.

**Giải thích từng đoạn.** Hàm `solve(lo, hi, optlo, opthi)` tính `cur[mid]`, ghi lại `best_j`, gọi hai nhánh. Khởi tạo `solve(t, n, t−1, n−1)`.

In [9]:
def dp_divide_conquer(nums, k):
    n = len(nums)
    S = prefix_sums(nums)
    INF = 1 << 120
    prev = [0] + [INF] * n
    for t in range(1, k + 1):
        cur = [INF] * (n + 1)

        def solve(lo, hi, optlo, opthi):
            if lo > hi:
                return
            mid = (lo + hi) // 2
            best, best_j = INF, optlo
            for j in range(optlo, min(mid - 1, opthi) + 1):
                v = prev[j] + (S[mid] - S[j]) ** 2
                if v < best:
                    best, best_j = v, j
            cur[mid] = best
            solve(lo, mid - 1, optlo, best_j)
            solve(mid + 1, hi, best_j, opthi)

        solve(t, n, t - 1, n - 1)
        prev = cur
    return (prev[n] + S[n]) // 2

for args, exp in EXAMPLES:
    got = dp_divide_conquer(*args)
    print(args, "->", got)
    assert got == exp

([5, 1, 2, 1], 2) -> 25
([1, 2, 3, 4], 1) -> 55
([1, 1, 1], 3) -> 3


**Độ phức tạp:** `O(k·n·log n)` thời gian, `O(n)` không gian (độ sâu đệ quy `O(log n)`). Nhanh hơn nhiều so với `O(k·n²)` nhưng khi `k` lớn (cỡ `n`) vẫn `O(n² log n)`.

### 7.3. Cách 4 — Aliens trick với DP có phạt `O(n²)`

**Mục đích.** Làm quen với Aliens trick bằng DP đơn giản trước khi tăng tốc bằng CHT.

**Giải thích từng đoạn.**

1. `penalized(S, λ)`: `dp[i] = min_j dp[j] + (S[i]−S[j])² + λ`, ghi thêm `cnt[i]` = số đoạn; **hòa** thì lấy `cnt` nhỏ hơn. Trả về `(dp[n], cnt[n])`.
2. Nhị phân `λ ∈ [0, S[n]²]` tìm `λ` nhỏ nhất với `cnt ≤ k` (tại `λ = S[n]²` luôn `cnt = 1`).
3. `F(k) = dp[n] − λ·k`; trả về `(F + S[n]) / 2`.

In [10]:
def penalized_quadratic(S, lam):
    n = len(S) - 1
    dp = [0] * (n + 1)
    cnt = [0] * (n + 1)
    for i in range(1, n + 1):
        best, best_c = None, None
        for j in range(i):
            v = dp[j] + (S[i] - S[j]) ** 2 + lam
            c = cnt[j] + 1
            if best is None or v < best or (v == best and c < best_c):
                best, best_c = v, c
        dp[i], cnt[i] = best, best_c
    return dp[n], cnt[n]

def aliens_quadratic(nums, k):
    S = prefix_sums(nums)
    lo, hi = 0, S[-1] ** 2
    while lo < hi:
        mid = (lo + hi) // 2
        _, c = penalized_quadratic(S, mid)
        if c <= k:
            hi = mid
        else:
            lo = mid + 1
    h, _ = penalized_quadratic(S, lo)
    F = h - lo * k
    return (F + S[-1]) // 2

for args, exp in EXAMPLES:
    got = aliens_quadratic(*args)
    print(args, "->", got)
    assert got == exp

([5, 1, 2, 1], 2) -> 25
([1, 2, 3, 4], 1) -> 55
([1, 1, 1], 3) -> 3


**Độ phức tạp:** `O(n² · log(S²))`: ví dụ `n = 100` rất nhanh; chỉ dùng để kiểm chứng và để hiểu Aliens.

### 7.4. Cách 5 — Aliens trick + Convex Hull Trick `O(n log(S²))`

**Giải thích từng đoạn.**

1. **Ghép đếm vào giá trị.** Đặt `W = 2^17 > n`. Thay mỗi số `dp[i]` bằng `D[i] = dp[i]·W + cnt[i]`. Chi phí một chuyển trạng thái `j → i` là `W·((S[i]−S[j])² + λ) + 1` (cộng `1` cho đếm). So sánh `D` chính là so sánh `(dp, cnt)` theo thứ tự từ điển.
2. **Đường thẳng cho `j`:** `slope = −2·W·S[j]`, `intercept = D[j] + W·S[j]²`. Giá trị tại `x = S[i]`: `slope·x + intercept`. Khi đó `D[i] = W·(S[i]² + λ) + 1 + min_j (đường thẳng j tại S[i])`.
3. **Truy vấn:** con trỏ `head` tiến khi đường thẳng kế tiếp cho giá trị `≤`.
4. **Thêm đường thẳng:** trước khi thêm, loại đường thẳng giữa `l2` nếu giao điểm `(l1, new)` nằm **không bên phải** giao điểm `(l1, l2)` — kiểm tra bằng nhân chéo số nguyên: `(b_new − b1)·(m1 − m2) ≤ (b2 − b1)·(m1 − m_new)`.
5. Cuối: `cost = D[n] // W`, `cnt = D[n] % W`.

In [11]:
W = 1 << 17

def penalized_cht(S, lam):
    n = len(S) - 1
    ms, bs = [0], [0]              # đường thẳng j = 0: slope -2*W*S[0] = 0, intercept D[0] + W*S[0]^2 = 0
    head = 0
    D = 0
    for i in range(1, n + 1):
        x = S[i]
        while head + 1 < len(ms) and ms[head + 1] * x + bs[head + 1] <= ms[head] * x + bs[head]:
            head += 1
        D = W * (x * x + lam) + 1 + ms[head] * x + bs[head]
        m_new, b_new = -2 * W * x, D + W * x * x
        while len(ms) - head >= 2:
            m1, b1 = ms[-2], bs[-2]
            m2, b2 = ms[-1], bs[-1]
            if (b_new - b1) * (m1 - m2) <= (b2 - b1) * (m1 - m_new):
                ms.pop(); bs.pop()
            else:
                break
        ms.append(m_new); bs.append(b_new)
    return D // W, D % W

def aliens_cht(nums, k):
    S = prefix_sums(nums)
    lo, hi = 0, S[-1] ** 2
    while lo < hi:
        mid = (lo + hi) // 2
        _, c = penalized_cht(S, mid)
        if c <= k:
            hi = mid
        else:
            lo = mid + 1
    h, _ = penalized_cht(S, lo)
    return (h - lo * k + S[-1]) // 2

for args, exp in EXAMPLES:
    got = aliens_cht(*args)
    print(args, "->", got)
    assert got == exp

([5, 1, 2, 1], 2) -> 25
([1, 2, 3, 4], 1) -> 55
([1, 1, 1], 3) -> 3


**Độ phức tạp:** `O(n)` mỗi `λ` (mỗi đường thẳng được thêm và bị loại tối đa một lần, con trỏ chỉ tiến), `~52` lần nhị phân ⇒ `O(n log(S²))` thời gian, `O(n)` không gian. Số nguyên Python tự mở rộng nên các phép nhân chéo (cỡ `10^40`) vẫn chính xác.

## 8. TRACE từng bước

### 8.1. Aliens trên `nums = [3,1,4,1,5]`, `k = 3`

Chạy hàm sau để xem với từng `λ` thử: `cost` (phần `Σ s²` có phạt), số đoạn `cnt`, và hướng thu hẹp khoảng. Quan sát `cnt` **không tăng** theo `λ`.

In [12]:
def trace_aliens(nums, k):
    S = prefix_sums(nums)
    print("S =", S, " T =", S[-1], " khoảng λ: [0,", S[-1] ** 2, "]")
    lo, hi = 0, S[-1] ** 2
    while lo < hi:
        mid = (lo + hi) // 2
        h, c = penalized_cht(S, mid)
        print(f"λ = {mid:4d}: h = {h:5d}, số đoạn nhỏ nhất cnt = {c}  ->", "cnt <= k: hi = mid" if c <= k else "cnt > k: lo = mid + 1")
        if c <= k:
            hi = mid
        else:
            lo = mid + 1
    h, c = penalized_cht(S, lo)
    F = h - lo * k
    print(f"λ* = {lo}: h = {h}, cnt = {c}, F(k) = h - λ*·k = {h} - {lo}*{k} = {F}")
    print("điểm = (F + T)/2 =", (F + S[-1]) // 2)
    return (F + S[-1]) // 2

nums, k = [3, 1, 4, 1, 5], 3
assert trace_aliens(nums, k) == brute(nums, k)
print("brute:", brute(nums, k), "| F(1..5):", dp_all_k(nums)[1:])

S = [0, 3, 4, 8, 9, 14]  T = 14  khoảng λ: [0, 196 ]
λ =   98: h =   294, số đoạn nhỏ nhất cnt = 1  -> cnt <= k: hi = mid
λ =   49: h =   198, số đoạn nhỏ nhất cnt = 2  -> cnt <= k: hi = mid
λ =   24: h =   138, số đoạn nhỏ nhất cnt = 3  -> cnt <= k: hi = mid
λ =   12: h =   102, số đoạn nhỏ nhất cnt = 3  -> cnt <= k: hi = mid
λ =    6: h =    82, số đoạn nhỏ nhất cnt = 4  -> cnt > k: lo = mid + 1
λ =    9: h =    93, số đoạn nhỏ nhất cnt = 3  -> cnt <= k: hi = mid
λ =    8: h =    90, số đoạn nhỏ nhất cnt = 3  -> cnt <= k: hi = mid
λ =    7: h =    86, số đoạn nhỏ nhất cnt = 4  -> cnt > k: lo = mid + 1
λ* = 8: h = 90, cnt = 3, F(k) = h - λ*·k = 90 - 8*3 = 66
điểm = (F + T)/2 = 40
brute: 40 | F(1..5): [196, 100, 66, 58, 52]


### 8.2. Convex hull: các đường thẳng được thêm / loại

Trace `penalized_cht` trên `S = [0, 3, 4, 8, 9, 14]` với `λ = 6`: sau mỗi `i` in `D[i]` (`cost`, `cnt`), con trỏ `head` và các đường thẳng còn trong bao.

In [13]:
def trace_cht(S, lam):
    n = len(S) - 1
    ms, bs = [0], [0]
    head = 0
    for i in range(1, n + 1):
        x = S[i]
        while head + 1 < len(ms) and ms[head + 1] * x + bs[head + 1] <= ms[head] * x + bs[head]:
            head += 1
        D = W * (x * x + lam) + 1 + ms[head] * x + bs[head]
        print(f"i={i}: x=S[i]={x}, đường thẳng tốt nhất là #{head} -> D = {D}  (cost = {D // W}, cnt = {D % W})")
        m_new, b_new = -2 * W * x, D + W * x * x
        popped = 0
        while len(ms) - head >= 2:
            m1, b1 = ms[-2], bs[-2]
            m2, b2 = ms[-1], bs[-1]
            if (b_new - b1) * (m1 - m2) <= (b2 - b1) * (m1 - m_new):
                ms.pop(); bs.pop(); popped += 1
            else:
                break
        ms.append(m_new); bs.append(b_new)
        print(f"      thêm đường thẳng j={i} (loại {popped} đường), bao hiện có {len(ms) - head} đường, head={head}")
    return D // W, D % W

S = [0, 3, 4, 8, 9, 14]
r = trace_cht(S, 6)
assert r == penalized_quadratic(S, 6)
print("kết quả (cost, cnt) =", r, "khớp DP O(n^2)")

i=1: x=S[i]=3, đường thẳng tốt nhất là #0 -> D = 1966081  (cost = 15, cnt = 1)
      thêm đường thẳng j=1 (loại 0 đường), bao hiện có 2 đường, head=0
i=2: x=S[i]=4, đường thẳng tốt nhất là #0 -> D = 2883585  (cost = 22, cnt = 1)
      thêm đường thẳng j=2 (loại 0 đường), bao hiện có 3 đường, head=0
i=3: x=S[i]=8, đường thẳng tốt nhất là #2 -> D = 5767170  (cost = 44, cnt = 2)
      thêm đường thẳng j=3 (loại 0 đường), bao hiện có 2 đường, head=2
i=4: x=S[i]=9, đường thẳng tốt nhất là #3 -> D = 6684675  (cost = 51, cnt = 3)
      thêm đường thẳng j=4 (loại 0 đường), bao hiện có 2 đường, head=3
i=5: x=S[i]=14, đường thẳng tốt nhất là #4 -> D = 10747908  (cost = 82, cnt = 4)
      thêm đường thẳng j=5 (loại 0 đường), bao hiện có 2 đường, head=4
kết quả (cost, cnt) = (82, 4) khớp DP O(n^2)


## 9. Code nộp LeetCode

Gộp Cách 5 (Aliens + CHT) thành `class Solution` độc lập. Tên phương thức suy luận — **cần đối chiếu** trên LeetCode.

In [14]:
from typing import List

class Solution:
    def minPartitionScore(self, nums: List[int], k: int) -> int:
        n = len(nums)
        S = [0] * (n + 1)
        for i, x in enumerate(nums):
            S[i + 1] = S[i] + x
        total = S[n]
        W = 1 << 17                       # > n: dùng để ghép (giá trị, số đoạn) thành một số nguyên

        def run(lam):
            # dp[i] = min_j dp[j] + (S[i]-S[j])^2 + lam, hòa -> số đoạn nhỏ nhất. Trả về (dp[n], cnt[n]).
            ms, bs = [0], [0]
            head = 0
            D = 0
            for i in range(1, n + 1):
                x = S[i]
                while head + 1 < len(ms) and ms[head + 1] * x + bs[head + 1] <= ms[head] * x + bs[head]:
                    head += 1
                D = W * (x * x + lam) + 1 + ms[head] * x + bs[head]
                mn = -2 * W * x
                bn = D + W * x * x
                while len(ms) - head >= 2:
                    m1, b1 = ms[-2], bs[-2]
                    m2, b2 = ms[-1], bs[-1]
                    if (bn - b1) * (m1 - m2) <= (b2 - b1) * (m1 - mn):
                        ms.pop()
                        bs.pop()
                    else:
                        break
                ms.append(mn)
                bs.append(bn)
            return D // W, D % W

        lo, hi = 0, total * total         # tại lam = total^2 luôn chỉ 1 đoạn
        while lo < hi:
            mid = (lo + hi) // 2
            if run(mid)[1] <= k:
                hi = mid
            else:
                lo = mid + 1
        h, _ = run(lo)
        return (h - lo * k + total) // 2

sol = Solution()
for args, exp in EXAMPLES:
    got = sol.minPartitionScore(*args)
    print(args, "->", got)
    assert got == exp

([5, 1, 2, 1], 2) -> 25
([1, 2, 3, 4], 1) -> 55
([1, 1, 1], 3) -> 3


## 10. Vì sao thuật toán đúng

**Rút gọn điểm.** `Σ s(s+1)/2 = (Σ s² + T)/2` với `T = Σ nums` cố định ⇒ chỉ cần `F(k) = min Σ s²`.

**Lồi.** Với cost đoạn `c(j, i) = (S[i] − S[j])²` ta có bất đẳng thức tứ giác `c(a, c) + c(b, d) ≤ c(a, d) + c(b, c)` với `a ≤ b ≤ c ≤ d` (khai triển: `−2·(S_c−S_a)... ` quy về `(S_b − S_a)(S_d − S_c) ≥ 0` do `S` tăng). Tính chất này suy ra `F` lồi theo `k` (kết quả kinh điển của quy hoạch động chia đoạn với cost Monge) — ta cũng đã kiểm chứng bằng số (mục 3.4 và test mục 12).

**Aliens.** Gọi `G(λ) = min_m (F(m) + λ m)` và `cnt(λ)` = số đoạn nhỏ nhất đạt `G(λ)`. Vì `F` lồi và nhận giá trị nguyên, các độ giảm `d_m = F(m) − F(m+1)` là số nguyên không tăng. `m` là tối ưu cho `λ` khi `d_m ≤ λ ≤ d_{m−1}`. Do đó tập số đoạn tối ưu tại `λ` là một đoạn liên tiếp `[cmin(λ), cmax(λ)]` và `cmax(λ) = cmin(λ − 1)` (vì các biên là số nguyên). Gọi `λ*` là giá trị nguyên nhỏ nhất với `cmin(λ*) ≤ k`; nếu `λ* = 0` thì `cmin(0) = n ≥ k` nên `k` thuộc đoạn tối ưu (số đoạn lớn nhất là `n`); ngược lại `cmin(λ* − 1) > k` nên `cmax(λ*) = cmin(λ* − 1) > k ≥ cmin(λ*)`, vậy `k ∈ [cmin(λ*), cmax(λ*)]` là số đoạn tối ưu tại `λ*`: `G(λ*) = F(k) + λ*·k` ⇒ `F(k) = G(λ*) − λ*·k`.

**Hòa.** Thuật toán lấy `cmin` bằng cách so sánh khóa `(dp, cnt)` theo thứ tự từ điển: `D = dp·W + cnt` với `cnt < W` biến so sánh từ điển thành so sánh số nguyên; phép cộng chi phí chuyển trạng thái `W·(cost + λ) + 1` cộng cả giá trị lẫn đếm; vì thế DP trên `D` tính đúng `(dp, cmin)` ở mọi `i` (một DP min trên nửa-vòng-số-nguyên mà phép cộng bảo toàn thứ tự).

**CHT.** `dp[j] + (S[i]−S[j])²` là giá trị của đường thẳng `j` tại `x = S[i]` cộng `S[i]²`; slope `−2WS[j]` giảm ngặt theo `j` (vì `S` tăng ngặt) và truy vấn `x = S[i]` tăng ngặt ⇒ bao dưới với con trỏ đơn điệu chính xác.

**Cận trên của `λ`.** Tại `λ = T²` hình phạt cho đoạn thứ hai vượt mọi mức tiết kiệm có thể (`F(1) = T²`), nên `cnt = 1 ≤ k`: nhị phân luôn có nghiệm.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`, `T = Σ nums ≤ 5·10^7`.

| Cách | Thời gian | Không gian |
|---|---|---|
| DP `O(k n²)` | `O(k·n²)` | `O(n)` |
| Chia để trị | `O(k·n log n)` | `O(n)` |
| Aliens + DP bình phương | `O(n² log T²)` | `O(n)` |
| **Aliens + CHT** | **`O(n log T²)`** | `O(n)` |

Với `n = 5·10^4`: `log₂(T²) ≈ 51` lần nhị phân, mỗi lần `5·10^4` bước CHT. Trong Python đo được ~ 2–3 giây (số nguyên lớn nên chậm hơn số 64-bit). Cận trên `λ` có thể thu hẹp thành `F(1) − F(2)`, nhưng `T²` đủ.

Đáp án có thể lên tới `~T²/2 ≈ 1.25·10^15` (khi `k = 1`) — vượt 32-bit nên ngôn ngữ khác cần `long long`; Python không tràn.

## 12. Bộ test

Ví dụ đề, edge case (`k = 1`, `k = n`, `n = 1`, mọi phần tử bằng nhau), kiểm chứng tính lồi, đối chiếu ngẫu nhiên giữa 6 cách có seed, và thời gian ở `n = 5·10^4`.

In [15]:
sol = Solution()
for args, exp in EXAMPLES:
    assert sol.minPartitionScore(*args) == exp

assert sol.minPartitionScore([7], 1) == 7 * 8 // 2                    # n = 1
assert sol.minPartitionScore([3, 3, 3, 3], 4) == 4 * (3 * 4 // 2)     # k = n: mỗi phần tử một đoạn
assert sol.minPartitionScore([2, 2, 2, 2], 2) == 2 * (4 * 5 // 2)     # chia đều: hai đoạn tổng 4
assert sol.minPartitionScore([1000] * 5, 1) == 5000 * 5001 // 2
print("ví dụ + edge case OK")

ví dụ + edge case OK


In [16]:
# Kiểm chứng tính lồi của F(m) trên nhiều mảng ngẫu nhiên (cơ sở của Aliens trick)
rng = random.Random(3929)
for _ in range(150):
    n = rng.randint(2, 14)
    a = [rng.randint(1, rng.choice([2, 10, 1000])) for _ in range(n)]
    F = dp_all_k(a)
    drops = [F[m] - F[m + 1] for m in range(1, n)]
    assert all(d > 0 for d in drops), a                     # F giảm ngặt
    assert all(drops[i] >= drops[i + 1] for i in range(len(drops) - 1)), (a, drops)   # lồi
print("F(m) giảm và lồi trên 150 mảng ngẫu nhiên")

F(m) giảm và lồi trên 150 mảng ngẫu nhiên


In [17]:
rng = random.Random(2026)
for _ in range(300):
    n = rng.randint(1, 10)
    k = rng.randint(1, n)
    a = [rng.randint(1, rng.choice([2, 9, 1000])) for _ in range(n)]
    exp = brute(a, k)
    assert dp_kn2(a, k) == exp, (a, k)
    assert dp_divide_conquer(a, k) == exp, (a, k)
    assert aliens_quadratic(a, k) == exp, (a, k)
    assert aliens_cht(a, k) == exp, (a, k)
    assert sol.minPartitionScore(a, k) == exp, (a, k)
print("300 test nhỏ: 6 cách khớp nhau")

for _ in range(150):
    n = rng.randint(10, 60)
    k = rng.randint(1, n)
    a = [rng.randint(1, rng.choice([3, 50, 1000])) for _ in range(n)]
    exp = dp_kn2(a, k)
    assert dp_divide_conquer(a, k) == exp, (a, k)
    assert aliens_quadratic(a, k) == exp, (a, k)
    assert sol.minPartitionScore(a, k) == exp, (a, k)
print("150 test vừa: DP = chia để trị = Aliens(bình phương) = Solution")

# mọi k trên một mảng, kể cả dữ liệu có nhiều "hòa" (mọi phần tử bằng nhau)
for a in ([4] * 12, [1, 1000] * 6, list(range(1, 14))):
    for k in range(1, len(a) + 1):
        assert sol.minPartitionScore(a, k) == dp_kn2(a, k), (a, k)
print("mọi k trên các mảng có nhiều hòa: khớp")

300 test nhỏ: 6 cách khớp nhau


150 test vừa: DP = chia để trị = Aliens(bình phương) = Solution
mọi k trên các mảng có nhiều hòa: khớp


In [18]:
# Thời gian ở giới hạn n = 5e4
rng = random.Random(1)
n = 50000
a = [rng.randint(1, 1000) for _ in range(n)]
for k in (1, 7, 1000, 25000, n):
    t0 = time.time()
    r = sol.minPartitionScore(a, k)
    print(f"n=5e4 k={k:6d} -> {r}  ({time.time() - t0:.2f}s)")
b = [1000] * n
t0 = time.time(); print("toàn 1000, k=3:", sol.minPartitionScore(b, 3), f"({time.time() - t0:.2f}s)")
# so sánh tốc độ với DP O(k n^2) ở cỡ nhỏ
a2 = a[:300]
t0 = time.time(); x1 = dp_kn2(a2, 30); t1 = time.time() - t0
t0 = time.time(); x2 = sol.minPartitionScore(a2, 30); t2 = time.time() - t0
assert x1 == x2
print(f"n=300, k=30: DP O(kn^2) {t1:.2f}s | Aliens+CHT {t2:.3f}s")

n=5e4 k=     1 -> 310975326753391  (1.81s)


n=5e4 k=     7 -> 44425057400406  (2.65s)


n=5e4 k=  1000 -> 311000793748  (2.62s)


n=5e4 k= 25000 -> 12820767111  (2.47s)


n=5e4 k= 50000 -> 8313384353  (2.26s)


toàn 1000, k=3: 416666692000000 (2.33s)


n=300, k=30: DP O(kn^2) 0.20s | Aliens+CHT 0.015s


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Bỏ ràng buộc "đúng `k` đoạn"

**Code sai:** DP không phạt, lấy `min` qua mọi số đoạn (hoặc `λ = 0`).
**Vì sao sai:** thêm đoạn luôn làm `Σ s²` giảm (mọi phần tử dương) nên tối ưu không ràng buộc luôn là `n` đoạn đơn lẻ.
**Code đúng:** ràng buộc đúng `k` (DP theo lớp) hoặc Aliens với `λ` được chọn để `k` là số đoạn tối ưu.

In [19]:
def wrong_ignore_k(nums, k):
    S = prefix_sums(nums)
    h, c = penalized_cht(S, 0)             # SAI: λ = 0 luôn cho n đoạn
    return (h + S[-1]) // 2

print("sai :", wrong_ignore_k([5, 1, 2, 1], 2), " đúng:", Solution().minPartitionScore([5, 1, 2, 1], 2))

sai : 20  đúng: 25


### Lỗi 2 — Khôi phục đáp án bằng số đoạn thực `cnt` thay vì `k`

**Code sai:** `F = h − λ·cnt`.
**Vì sao sai:** khi hòa, `cnt` (số đoạn nhỏ nhất) có thể `< k` (nhiều số đoạn cùng tối ưu); công thức `F(k) = h − λk` dùng **`k`**, vì `k` cũng tối ưu tại `λ*`.
**Code đúng:** `h − λ·k`.

In [20]:
def wrong_use_cnt(nums, k):
    S = prefix_sums(nums)
    lo, hi = 0, S[-1] ** 2
    while lo < hi:
        mid = (lo + hi) // 2
        if penalized_cht(S, mid)[1] <= k:
            hi = mid
        else:
            lo = mid + 1
    h, c = penalized_cht(S, lo)
    return (h - lo * c + S[-1]) // 2              # SAI: dùng c thay cho k

rng = random.Random(4)
for _ in range(400):
    n = rng.randint(3, 8); k = rng.randint(1, n)
    a = [rng.choice([1, 2, 3]) for _ in range(n)]
    if wrong_use_cnt(a, k) != dp_kn2(a, k):
        print("phản ví dụ:", a, k)
        print("sai :", wrong_use_cnt(a, k), " đúng:", dp_kn2(a, k))
        break

phản ví dụ: [1, 3, 1, 3, 2, 3, 1] 6
sai : 33  đúng: 27


### Lỗi 3 — Cận trên `λ` quá nhỏ

**Code sai:** `hi = 10**9`.
**Vì sao sai:** độ giảm `F(1) − F(2)` có thể cỡ `T²/2 ≈ 10^15`. Với `hi` nhỏ, nhị phân dừng ở `λ` mà số đoạn vẫn `> k`.
**Code đúng:** `hi = T²` (hoặc `F(1)`).

In [21]:
def wrong_small_hi(nums, k, hi_cap=10**4):
    S = prefix_sums(nums)
    lo, hi = 0, hi_cap                            # SAI: cận trên cố định nhỏ
    while lo < hi:
        mid = (lo + hi) // 2
        if penalized_cht(S, mid)[1] <= k:
            hi = mid
        else:
            lo = mid + 1
    h, _ = penalized_cht(S, lo)
    return (h - lo * k + S[-1]) // 2

a = [1000] * 40
print("sai :", wrong_small_hi(a, 1), " đúng:", Solution().minPartitionScore(a, 1), "(k = 1 cần λ cỡ T^2/2 ≈ 8·10^8)")

sai : 20215000  đúng: 800020000 (k = 1 cần λ cỡ T^2/2 ≈ 8·10^8)


### Lỗi 4 — Quên chia đôi (`(Σ s² + T)/2`)

Nếu trả về `F(k)` hoặc `F(k) + T` mà không chia 2 thì sai. Kiểm tra nhanh Ví dụ 3: `Σ s² = 3`, `T = 3` → `(3 + 3)/2 = 3`.

### Lỗi 5 — Dùng số thực trong CHT

Giao điểm hai đường thẳng tính bằng `/` cho `float` mất chính xác với số cỡ `10^15`–`10^20`; dùng **nhân chéo trên số nguyên**.

### Lỗi 6 — DP `O(k·n²)` cho `n = 5·10^4`

Chỉ để kiểm chứng; xem so sánh thời gian ở mục 12.

In [22]:
# Lỗi 5: tìm (có seed) một trường hợp mà so sánh giao điểm bằng float cho kết quả KHÁC so sánh nhân chéo số nguyên
rng = random.Random(5)
found = None
for _ in range(100000):
    d1 = rng.randint(10**9, 10**10); d2 = d1 + rng.randint(1, 10**9)      # d1 = m1 - m2, d2 = m1 - m3
    n2 = rng.randint(10**16, 10**17)                                     # n2 = b2 - b1
    n3 = (n2 * d2) // d1 + rng.randint(-1, 1)                            # n3 = b3 - b1, gần giao điểm
    exact = n3 * d1 <= n2 * d2                                           # nhân chéo số nguyên: chính xác
    approx = (n3 / d2) <= (n2 / d1)                                      # chia số thực
    if exact != approx:
        found = (d1, d2, n2, n3, exact, approx)
        break
print("(m1-m2, m1-m3, b2-b1, b3-b1, chính xác, float) =", found)
assert found is not None

(m1-m2, m1-m3, b2-b1, b3-b1, chính xác, float) = (1673671309, 2072881389, 64872997153957288, 80346736976350970, False, True)


## 14. Mẹo & nhận dạng pattern

- **"Chia mảng thành đúng `k` đoạn, cost đoạn là hàm lồi của tổng"** → hai công cụ: **tối ưu chia để trị** (`O(k n log n)`) hoặc **Aliens trick** (`O(n log C)` nếu DP có phạt tăng tốc được).
- **Aliens trick cần:** hàm `F(k)` **lồi** (hoặc lõm nếu tối đa) + một DP không ràng buộc nhanh. Thay "đúng `k`" bằng phạt `λ` mỗi đoạn, nhị phân `λ` trên số nguyên, khôi phục `F(k) = h − λ·k`.
- **Xử lý hòa:** luôn nhất quán — ở đây lấy số đoạn **nhỏ nhất**, tìm `λ` **nhỏ nhất** có `cnt ≤ k`; nên ghép đếm vào giá trị (`D = dp·W + cnt`) để so sánh một lần.
- **CHT** xuất hiện khi chuyển trạng thái có dạng `dp[j] + (a_i − a_j)²` → mở bình phương thành tích `a_i·a_j` + hạng tử riêng: `y = m·x + b`.
- Bài giá trị `s(s+1)/2` có thể quy về `Σ s²` nhờ **tổng các `s` không đổi** — đừng cố tối ưu trực tiếp.
- Số lớn (`10^15`–`10^40`): dùng số nguyên Python và **nhân chéo**; tránh `float`.

## 15. Biến thể

1. **Phiên bản I (nhỏ hơn):** `n ≤ 1000`, `k ≤ 1000` — DP `O(k·n²)` hoặc chia để trị là đủ.
2. **Cost `s³` hoặc `s^p`:** vẫn lồi → Aliens dùng được nhưng CHT không (cần chia để trị / Li Chao tùy dạng).
3. **Tối đa thay vì tối thiểu:** cost lõm (như `√s`) → `F` lõm theo `k`; đổi dấu rồi dùng cùng khung.
4. **Số đoạn không quá `k`:** vì `F` giảm, "tối đa `k` đoạn" tương đương "đúng `k` đoạn" (nên dùng tối đa `min(k, n)` đoạn).
5. **Khôi phục cách chia:** theo dõi `prev[i]` trong DP có phạt và quay lui — cần chú ý hòa để ra đúng `k` đoạn (kỹ thuật phụ phức tạp hơn).

In [23]:
# Biến thể 4: "tối đa k đoạn" cho cùng đáp án với "đúng k đoạn" (vì F giảm ngặt)
rng = random.Random(15)
for _ in range(100):
    n = rng.randint(1, 10)
    a = [rng.randint(1, 20) for _ in range(n)]
    k = rng.randint(1, n)
    at_most = min(brute(a, m) for m in range(1, k + 1))
    assert at_most == brute(a, k)
print("tối đa k đoạn = đúng k đoạn (vì thêm đoạn luôn có lợi)")

# Biến thể 2: cost s^3 vẫn lồi theo k (chỉ để minh họa)
def F_cubic(a):
    n = len(a); S = prefix_sums(a); INF = float("inf")
    dp = [0] + [INF] * n; out = [None]
    for t in range(1, n + 1):
        nd = [INF] * (n + 1)
        for i in range(t, n + 1):
            nd[i] = min(dp[j] + (S[i] - S[j]) ** 3 for j in range(t - 1, i))
        dp = nd; out.append(dp[n])
    return out
Fc = F_cubic([3, 1, 4, 1, 5, 9, 2, 6])
dr = [Fc[m] - Fc[m + 1] for m in range(1, len(Fc) - 1)]
print("độ giảm với cost s^3:", dr, "-> không tăng:", all(dr[i] >= dr[i + 1] for i in range(len(dr) - 1)))

tối đa k đoạn = đúng k đoạn (vì thêm đoạn luôn có lợi)
độ giảm với cost s^3: [22134, 3672, 2016, 414, 288, 60, 36] -> không tăng: True


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi tập điểm cắt | `O(C(n−1, k−1)·n)` | `O(n)` | kiểm chứng, `n ≤ 14` |
| 2. DP `O(k n²)` | `dp[t][i]` đầy đủ | `O(k·n²)` | `O(n)` | `n ≲ 500` |
| 3. Chia để trị | `opt[i]` đơn điệu | `O(k·n log n)` | `O(n)` | khi `k` nhỏ (≲ 100) hoặc `n` vừa |
| 4. Aliens + DP bình phương | phạt `λ` + nhị phân `λ` | `O(n² log T²)` | `O(n)` | hiểu Aliens; `n ≲ 1000` |
| 5. Aliens + CHT | như 4, DP có phạt bằng CHT | `O(n log T²)` | `O(n)` | **lời giải nộp**; `n = 5·10^4`, mọi `k` |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `score_direct([[2, 2], [1]])` bằng bao nhiêu?

**Bài 2 (tính tay).** `nums = [1, 1, 1, 1]`, `k = 2`. Điểm nhỏ nhất là bao nhiêu? (Chia thành hai đoạn tổng `2` và `2`.)

**Bài 3 (điền TODO).** Hoàn thiện `my_line_value(S, F, j, x)` — giá trị đường thẳng `j` tại `x`: `−2·S[j]·x + F[j] + S[j]²` (ở đây `F[j] = dp[j]`).

**Bài 4 (tự viết lại).** Viết `my_min_partition_score(nums, k)` theo Cách 5 (Aliens + CHT), không nhìn code nộp.

**Bài 5 (biến thể).** Viết `max_partition_score(nums, k)`: **lớn nhất** điểm khi chia đúng `k` đoạn (gợi ý: `Σ s²` lớn nhất ở cách chia lệch nhất; vét cạn cho `n ≤ 10` đủ).

In [24]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_line_value(S, F, j, x):
    # TODO: trả về -2*S[j]*x + F[j] + S[j]**2
    pass

# Bài 4 — TODO
def my_min_partition_score(nums, k):
    # TODO: S; W = 1 << 17; penalized(lam) bằng CHT; nhị phân lam trong [0, T*T]; F = h - lam*k; (F + T)//2
    pass

# Bài 5 — TODO
def max_partition_score(nums, k):
    # TODO: combinations các điểm cắt, tính điểm bằng định nghĩa, lấy max
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tổng từng đoạn `4` và `1`.
- Mức 2: `4·5/2 = 10`, `1·2/2 = 1`.
- Mức 3: `11`.

**Bài 2.**
- Mức 1: các `s` đều nhau (`2` và `2`) cho `Σ s²` nhỏ nhất.
- Mức 2: mỗi đoạn `2·3/2 = 3`.
- Mức 3: `3 + 3 = 6`.

**Bài 3.**
- Mức 1: viết đúng công thức đường thẳng ở mục 3.5.
- Mức 2: slope `−2·S[j]`, hệ số tự do `F[j] + S[j]²`.
- Mức 3: `return -2*S[j]*x + F[j] + S[j]*S[j]`.

**Bài 4.**
- Mức 1: xem lại `class Solution` ở mục 9.
- Mức 2: `D = dp·W + cnt`; hòa lấy `cnt` nhỏ.
- Mức 3: `F(k) = h − λ·k`, đáp án `(F + T)//2`.

**Bài 5.**
- Mức 1: `combinations(range(1, n), k - 1)` như `brute`.
- Mức 2: thay `min` bằng `max`.
- Mức 3: `[1,1,1,1]`, `k = 2`: chia `[1,1,1] | [1]` → `6 + 1 = 7`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert score_direct([[2, 2], [1]]) == 11
# Bài 2
assert Solution().minPartitionScore([1, 1, 1, 1], 2) == 6 == brute([1, 1, 1, 1], 2)

# Bài 3
def my_line_value_ans(S, F, j, x):
    return -2 * S[j] * x + F[j] + S[j] * S[j]

S = [0, 5, 6, 8, 9]
Fj = [0, 25, 36, 64, 81]                 # dp[j] = S[j]^2 (một đoạn)
for j in range(5):
    for i in range(j + 1, 5):
        assert my_line_value_ans(S, Fj, j, S[i]) + S[i] ** 2 == Fj[j] + (S[i] - S[j]) ** 2
print("Bài 1–3 OK")

Bài 1–3 OK


In [26]:
# Bài 4
def my_min_partition_score_ans(nums, k):
    n = len(nums)
    S = prefix_sums(nums)
    T = S[-1]
    W2 = 1 << 17

    def penalized(lam):
        ms, bs, head, D = [0], [0], 0, 0
        for i in range(1, n + 1):
            x = S[i]
            while head + 1 < len(ms) and ms[head + 1] * x + bs[head + 1] <= ms[head] * x + bs[head]:
                head += 1
            D = W2 * (x * x + lam) + 1 + ms[head] * x + bs[head]
            mn, bn = -2 * W2 * x, D + W2 * x * x
            while len(ms) - head >= 2 and (bn - bs[-2]) * (ms[-2] - ms[-1]) <= (bs[-1] - bs[-2]) * (ms[-2] - mn):
                ms.pop(); bs.pop()
            ms.append(mn); bs.append(bn)
        return D // W2, D % W2

    lo, hi = 0, T * T
    while lo < hi:
        mid = (lo + hi) // 2
        if penalized(mid)[1] <= k:
            hi = mid
        else:
            lo = mid + 1
    h, _ = penalized(lo)
    return (h - lo * k + T) // 2

rng = random.Random(29)
for _ in range(100):
    n = rng.randint(1, 30); k = rng.randint(1, n)
    a = [rng.randint(1, 100) for _ in range(n)]
    assert my_min_partition_score_ans(a, k) == dp_kn2(a, k)

# Bài 5
def max_partition_score_ans(nums, k):
    n = len(nums)
    best = None
    for cuts in combinations(range(1, n), k - 1):
        b = (0,) + cuts + (n,)
        sc = sum(sum(nums[b[i]:b[i + 1]]) * (sum(nums[b[i]:b[i + 1]]) + 1) // 2 for i in range(k))
        best = sc if best is None else max(best, sc)
    return best

assert max_partition_score_ans([1, 1, 1, 1], 2) == 7
assert max_partition_score_ans([5, 1, 2, 1], 2) == 37
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (Aliens + CHT, bài min với cost lồi):**

```python
S = prefix sums; T = S[n]; W = 1 << 17          # W > n
def run(lam):                                     # trả (cost, cnt nhỏ nhất)
    lines: slope = -2*W*S[j], intercept = D[j] + W*S[j]**2
    D[i] = W*(S[i]**2 + lam) + 1 + min_j(line_j(S[i]))     # con trỏ đơn điệu
    thêm đường thẳng: pop khi (b_new-b1)*(m1-m2) <= (b2-b1)*(m1-m_new)
    return D[n] // W, D[n] % W
lo, hi = 0, T*T
while lo < hi: mid; hi = mid if run(mid)[1] <= k else lo = mid + 1   # λ nhỏ nhất có cnt <= k
F = run(lo)[0] - lo * k
return (F + T) // 2
```

**Checklist trước khi submit:**

- [ ] Quy điểm về `Σ s²`: đáp án `(F + T)//2`.
- [ ] Tìm `λ` **nhỏ nhất** có `cnt ≤ k`, `cnt` là **số đoạn nhỏ nhất** khi hòa.
- [ ] Khôi phục `F = h − λ·k` (dùng `k`, không dùng `cnt`).
- [ ] Cận trên `hi = T²` (đủ lớn); `lo = 0`.
- [ ] Ghép `(giá trị, đếm)` bằng `W > n`; phép nhân chéo bằng số nguyên (không float).
- [ ] Đường thẳng thêm theo slope giảm dần, truy vấn `x = S[i]` tăng dần (con trỏ `head` chỉ tiến).
- [ ] `k = n` cho `λ = 0` (đáp án `Σ nums[i](nums[i]+1)/2`); `k = 1` cho `λ` rất lớn.
- [ ] Đã đối chiếu vét cạn `n ≤ 10` và DP `O(k n²)` tới `n ≈ 60`, kể cả mảng nhiều "hòa".
- [ ] Đã đối chiếu chữ ký phương thức Premium trên LeetCode.